# Module 9: Quantization & Efficient Inference

## Learning Objectives
- Explain quantization tradeoffs (FP8, AWQ, GGUF)
- Quantize a small model and compare outputs
- Observe accuracy degradation
- Connect to Kyro's quantized model loader

## Concept: Quantization

Reducing weight precision reduces memory and increases throughput:
| Format | Bits | Memory | Speed | Quality |
|--------|------|--------|-------|----------|
| FP32 | 32 | 1x | 1x | Full |
| FP16 | 16 | 0.5x | ~1.5x | Near-full |
| INT8 | 8 | 0.25x | ~2x | Slight loss |
| INT4 | 4 | 0.125x | ~3x | Noticeable loss |

**Kyro mapping**: `src/model/quantized.rs::QuantizedLlama`

In [ ]:
import numpy as np
np.random.seed(42)
w = np.random.randn(64, 64).astype(np.float32)
print(f'FP32: {w.nbytes} bytes')

w_fp16 = w.astype(np.float16)
print(f'FP16: {w_fp16.nbytes} bytes ({w.nbytes/w_fp16.nbytes:.1f}x)')

scale = np.max(np.abs(w))
w_i8 = (w / scale * 127).astype(np.int8)
w_d8 = (w_i8.astype(np.float32) / 127 * scale)
print(f'INT8: {w_i8.nbytes} bytes ({w.nbytes/w_i8.nbytes:.1f}x)')

w_i4 = (w / scale * 7).astype(np.int8)
w_d4 = (w_i4.astype(np.float32) / 7 * scale)
print(f'INT4: {w_i4.nbytes} bytes ({w.nbytes/w_i4.nbytes:.1f}x)')

print(f'Errors: FP16={np.abs(w-w_fp16.astype(np.float32)).mean():.6f}, INT8={np.abs(w-w_d8).mean():.6f}, INT4={np.abs(w-w_d4).mean():.6f}')

## Exercises
1. Experiment with 4-bit quantization on a small transformer.
2. Compare output degradation across quantization levels.
3. Explore Kyro's `src/model/quantized.rs` and `src/model/loader.rs`.